In [1]:
! --------------------------------------------------------------------------
! PARAMETERS
! --------------------------------------------------------------------------
integer, parameter :: Nx = 256+2         ! Number of grid points in x
integer, parameter :: Ny = 256+2         ! Number of grid points in y
double precision, parameter :: tol = 1.0d-9
double precision, parameter :: w   = 1.99d0  ! Over-relaxation factor (1 < w < 2) WARNIG: it is case dependent!
integer, parameter :: maxIter = 100000

! --------------------------------------------------------------------------
! VARIABLES
! --------------------------------------------------------------------------
  double precision, dimension(Nx, Ny) :: u
  double precision :: dx, dy, x, y
  double precision :: err, diff, uNew, uOld, start, finish, exactval, maxerr, pi
  integer :: i, j, iter

In [2]:
call cpu_time(start)
! --------------------------------------------------------------------------
! COMPUTE GRID SPACING
! --------------------------------------------------------------------------
pi = acos(-1.0d0)
dx = 1.0d0 / dble(Nx - 1)
dy = 1.0d0 / dble(Ny - 1)

! --------------------------------------------------------------------------
! INITIALIZE SOLUTION AND APPLY BOUNDARY CONDITIONS
! --------------------------------------------------------------------------
u = 0.0d0

! Along x=0 and x=1 => u=0 (already zero-initialized)
! y=0 => u(x,0) = sin(pi*x)
do i = 0, Nx-1
    x = dble(i) * dx
    u(i+1,1) = sin(pi*x)
end do

! y=1 => u(x,1) = sin(pi*x)*exp(-pi)
do i = 0, Nx-1
    x = dble(i) * dx
    u(i+1,Ny) = sin(pi*x)*exp(-pi)
end do

! --------------------------------------------------------------------------
! SOR ITERATION
! --------------------------------------------------------------------------
do iter = 1, maxIter
    err = 0.0d0

    ! Update interior points (i=2..Nx-1, j=2..Ny-1)
    do j = 2, Ny-1
       do i = 2, Nx-1
          uOld = u(i,j)

          ! Laplace update (5-point stencil, uniform dx=dy)
          uNew = 0.25d0*( u(i+1,j) + u(i-1,j) + &
                          u(i,j+1) + u(i,j-1) )

          ! SOR update
          diff   = uNew - uOld
          u(i,j) = uOld + w*diff

          ! Track maximum update
          if (abs(diff) > err) err = abs(diff)
       end do
end do

! Check for convergence
if (err < tol) then
       print *, 'Converged after ', iter, ' iterations, err=', err
       exit
end if

end do

if (iter > maxIter) then
     print *, 'Warning: reached maxIter without full convergence.'
end if

Converged after     1981     iterations, err=    9.91502548466915989e-10


In [3]:
!-----------------------------------------------------------------------
! Compare numerical solution to exact solution: sin(pi*x)*exp(-pi*y)
!-----------------------------------------------------------------------
maxErr = 0.0d0
do j = 1, Ny
    y = (j - 1)*dy
    do i = 1, Nx
      x = (i - 1)*dx
      exactVal = sin(pi*x)*exp(-pi*y)
      diff = dabs(u(i,j) - exactVal)
      if (diff > maxErr) maxErr = diff
    end do
end do

print *, "Max difference from exact solution = ", maxErr
print *, "Potential at (x=0.5,y=0.5) ~ ", u((Nx-1)/2, (Ny-1)/2)

call cpu_time(finish)
print '("Time = ",f10.4," seconds.")',finish-start

Max difference from exact solution =     4.41315253163487853e-06
Potential at (x=0.5,y=0.5) ~     2.11694617035636246e-01
Time =     0.9361 seconds.
